# HealthService MOCK: the dashboard for the 3D page

This is a **mock** of the HealthService, which another part of the team builds. It lives in `sim/mocks/healthservice/` and is not part of the DataService. It reads from gravsim's DataService and builds the JSON the 3D page reads (contract version 1, `docs/frontend-backend.md` on the `design-body` branch). It uses only what the instruments reported, never gravsim's hidden truth.

Everything below is **synthetic**, from curves fitted to NASA bed-rest studies, not spaceflight.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))     # sim/, when run from sim/mocks/healthservice
import pandas as pd
import gravsim as gs
from mocks.healthservice import HealthService, validate_dashboard

mission = gs.Mission("iss_180", launch_date="2027-03-01")      # the four crew of examples/gravsim_demo.ipynb
ana = mission.add_crew("Ana", sex="F", adherence=1.0, responder=0, seed=1)
ben = mission.add_crew("Ben", sex="M", adherence=0.0, responder=0, seed=2)
caro = mission.add_crew("Caro", sex="F", adherence=[(0, 1.0), (60, 0.2), (120, 1.0)], seed=3)
caro.add_event(day=95, measures="legs", severity_pct=12, recovery_days=20, label="ankle sprain")
dani = mission.add_crew("Dani", sex="M", adherence=1.0, responder="fast", seed=4)
data = gs.DataService()
data.ingest(mission.timeline(seed=10))
print(len(data.readings("Caro")), "readings for Caro")

1212 readings for Caro


Caro's dashboard on flight day 111, during her two months of skipped exercise and two weeks after her ankle sprain. Scores colour the body: 0 is the pre-flight reference, 100 is as much loss as bed-rest subjects had with no exercise at all. It is a visual scale, not a clinical grade.

In [2]:
dash = HealthService(data).dashboard(caro, day=110)
validate_dashboard(dash)                  # the same checks the 3D page runs
pd.DataFrame(dash["summary"]["items"])[["title", "changePercent", "description"]]

,title,changePercent,description
0,Quadriceps,-7.0,Knee extension strength at 60°/s (dynamometer)...
1,Hamstrings,-15.6,"Knee flexion strength at 60°/s (dynamometer), ..."
2,Calves,-37.7,Ankle plantarflexion strength at 30°/s (dynamo...
3,Shins,-15.4,Ankle dorsiflexion strength at 30°/s (dynamome...
4,Back,-24.0,Trunk extension strength at 60°/s (dynamometer...
5,Arms,NaN,"Arm lean mass (DXA), latest L-60, compared wit..."


In [3]:
print(dash["summary"]["title"], "|", dash["summary"]["notice"])
print("scores:", dash["body"]["scores"])
print("insight:", dash["charts"][0]["insight"])

Caro · FD111 | Synthetic data from gravsim 0.2.0, built on NASA bed-rest studies. Not a health assessment.
scores: {'quadriceps_l': 29.8, 'quadriceps_r': 29.8, 'hamstrings_l': 100.0, 'hamstrings_r': 100.0, 'calf_l': 100.0, 'calf_r': 100.0, 'shin_muscles_l': 100.0, 'shin_muscles_r': 100.0, 'back_muscles_l': 100.0, 'back_muscles_r': 100.0}
insight: -7% since pre-flight. After 110 days without gravity, bed-rest subjects changed about -23% without exercise and -8% with it.


Every dashboard is saved back through the DataService as two processed results, and the DataService announces each one on the event queue (its default publisher is a mock that keeps the messages).

In [4]:
print([(r.kind, r.timestamp) for r in data.results("Caro")])
pd.DataFrame([{"topic": e["topic"], **e["message"]} for e in data.publisher.messages if e["topic"] == "data.result_saved"])

[('dashboard', '2027-06-19T12:00:00Z'), ('measure_change', '2027-06-19T12:00:00Z')]


,topic,crew_id,kind,timestamp,computed_at,model_version
0,data.result_saved,Caro,dashboard,2027-06-19T12:00:00Z,2026-11-14T21:51:13Z,0.2.0
1,data.result_saved,Caro,measure_change,2027-06-19T12:00:00Z,2026-11-14T21:51:13Z,0.2.0


To feed the 3D page, run the mock's server from `sim/` and point the page at it:

```bash
python -m mocks.healthservice.serve --day 95 --speed 20 --port 3000
# then, in src/front/sketch on the design-body branch:
VITE_DASHBOARD_URL=http://localhost:3000/api/v1/dashboard npm run dev
```